# A2C on CartPole-v1

**Advantage Actor-Critic (A2C)** is a synchronous, on-policy algorithm that combines:

- **Actor**: a policy network that outputs action probabilities.
- **Critic**: a value network that estimates the state value $V(s)$.

The advantage $A(s, a) = R - V(s)$ reduces the variance of the policy gradient update.

> Paper: [Asynchronous Methods for Deep Reinforcement Learning](https://arxiv.org/abs/1602.01783)

In this notebook we train A2C on the classic **CartPole-v1** task (discrete action space) using the `lit-sb3` SB3-compatible API, which is backed by PyTorch Lightning.


## Setup

Make sure `lit-sb3` and `gymnasium` are installed in your environment.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

from lit_sb3.algorithms.a2c import A2C

## Create the model

We use the `MlpPolicy` string shorthand. Custom network architecture is passed via `policy_kwargs`.
Key hyperparameters:

| Parameter    | Description                                         |
| ------------ | --------------------------------------------------- |
| `n_steps`    | Steps collected per env per update (rollout length) |
| `gamma`      | Discount factor                                     |
| `gae_lambda` | GAE lambda; `1.0` = standard advantage              |
| `ent_coef`   | Entropy bonus to encourage exploration              |
| `vf_coef`    | Weight of the value-function loss                   |


In [ ]:
model = A2C(
    "MlpPolicy",
    "CartPole-v1",
    verbose=1,
    n_steps=5,
    learning_rate=7e-4,
    gamma=0.99,
    gae_lambda=1.0,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={"net_arch": {"pi": [64, 64], "vf": [64, 64]}},
)
print(model.policy)

## Train

`learn()` translates `total_timesteps` into Lightning `max_epochs` and calls `Trainer.fit` internally.


In [ ]:
model.learn(total_timesteps=10_000)

## Evaluate

Run the trained policy for a few episodes and report the mean episode return.


In [ ]:
eval_env = gym.make("CartPole-v1")
n_eval_episodes = 10
episode_rewards = []

for _ in range(n_eval_episodes):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    while not done:
        action, _ = model.predict(obs[np.newaxis], deterministic=True)
        obs, reward, terminated, truncated, _ = eval_env.step(int(action[0]))
        total_reward += reward
        done = terminated or truncated
    episode_rewards.append(total_reward)

print(f"Mean reward over {n_eval_episodes} episodes: {np.mean(episode_rewards):.1f} (± {np.std(episode_rewards):.1f})")